# North Carolina Housing: Main Project Notebook

**Team:** Stephen Dillon, Josiah Bradshaw, Kyle Hampton, Allison Haynes  
**Source snapshot:** four notebooks in the uploaded repository ZIP, dated November 1, 2026.

**Central question:** Where in North Carolina can households with different budgets and preferences find affordable housing with access to the amenities and services they value?

This notebook contains the source preparation, geographic integration, accessibility features, and presentation maps in one ordered workflow. The originals used separate sessions; this version has shared paths, explicit rebuild switches, and a final exported feature table.

## How to run

1. Put this notebook in the project root, beside `data/`.
2. Install the packages in the commented setup line if needed. Select that Python environment as the notebook kernel.
3. For normal team use, keep the rebuild switches **False** and copy the shared `data/processed/` folder into place. This route loads saved preparation results, then recalculates the accessibility features and exports the combined table.
4. To rerun original source preparation, turn on the relevant rebuild switch. Raw files are listed below. Set `REBUILD_AREAS=True` to rebuild the tract/ZCTA intersections and EPA assignment. Boundary files are cached; a first rebuild can download a roughly 504 MB national ZCTA file.
5. Restart the kernel and run all cells in order. Network sources and excluded local data are required only for stages being rebuilt.

### Workflow

- 1. Setup and file checks
- 2. ACS extraction and cleaning
- 3. Zillow and EPA cleaning and population-weighted ZIP export
- 4. OSM, PAD-US, coastal access, rail/bus, and airport preparation
- 5. Tract/ZCTA integration and dominant EPA assignment
- 6. Amenity counts and nearest-access distances
- 7. Validation, exports, maps, and descriptive summaries
- 8. Limitations and remaining analysis

The ZIP contains notebooks but no raw or processed data. No past cell outputs are carried into this notebook as fresh results.

## 1. Shared setup

| Stage | Raw inputs for a rebuild |
|---|---|
| ACS | `data/acs5_2024_tracts_raw.csv` cache, or Census API access |
| Zillow/EPA | `data/zillowzipcode.csv`, `data/SmartLocationDatabase.gdb`; Census 2010 crosswalk is downloaded if absent |
| Amenities | `data/north-carolina.gpkg`, `data/PADUS4_1_StateNC.gdb`; coastal/rail APIs need internet |
| Geographic areas | Four processed housing CSVs and cached/downloadable Census boundaries |

A Census key is read from `CENSUS_API_KEY` in the environment, never embedded in this file. Historical nationwide ACS maps remain available through `DRAW_SOURCE_MAPS`; they run only when rebuilding ACS. Folium is optional.

In [71]:
# Uncomment once if packages are missing, then restart the kernel.
# %pip install pandas numpy requests geopandas pyogrio matplotlib
# Optional interactive map only: %pip install folium

import os
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import geopandas as gpd
import pyogrio
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch, Rectangle
from shapely.geometry import box, Point
from IPython.display import display

PROJECT_DIR = Path.cwd()  # Change this if your kernel starts outside the project.
DATA_DIR = PROJECT_DIR / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
BOUNDARY_DIR = DATA_DIR / 'raw' / 'boundaries'
FIG_DIR = PROJECT_DIR / 'figures'
OUTPUT_DIR = PROCESSED_DIR
for folder in [DATA_DIR, PROCESSED_DIR, BOUNDARY_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

REBUILD_ACS = False
REBUILD_ZILLOW_EPA = False
REBUILD_AMENITIES = False
REBUILD_AREAS = False
DRAW_SOURCE_MAPS = False
DRAW_PRESENTATION_MAPS = False
DRAW_INTERACTIVE_MAP = False
SEARCH_RADIUS_M = 5000
PROJECT_CRS = 'EPSG:32119'
YEAR = 2024
CENSUS_API_KEY = os.environ.get('CENSUS_API_KEY', '')
OSM_PATH = DATA_DIR / 'north-carolina.gpkg'
PADUS_PATH = DATA_DIR / 'PADUS4_1_StateNC.gdb'
pd.set_option('display.max_columns', 50)
print('Project folder:', PROJECT_DIR)

Project folder: /Users/josiah/2302GroupProj/2302-group-proj


In [72]:
def require_files(paths, purpose):
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(purpose + '\nMissing:\n' + '\n'.join(missing))

def download_cached(url, path):
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.part')
        print('Downloading:', path.name)
        with requests.get(url, stream=True, timeout=(30, 180)) as response:
            response.raise_for_status()
            with temporary.open('wb') as file:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        file.write(chunk)
        temporary.replace(path)
    return path

def save_gpkg(frame, filename, layer):
    saved = frame.copy()
    for column in saved.columns:
        if isinstance(saved[column].dtype, pd.CategoricalDtype):
            saved[column] = saved[column].astype('string')
    path = PROCESSED_DIR / filename
    path.unlink(missing_ok=True)
    saved.to_file(path, layer=layer, driver='GPKG', engine='pyogrio')

expected = ['nc_acs_2024_tract_clean.csv', 'nc_zillow_zip_clean.csv',
            'nc_epa_2010_tract_clean.csv', 'nc_zillow_epa_zip_clean.csv',
            'osm_amenities_clean.gpkg', 'padus_open_clean.gpkg',
            'nc_coastal_access_clean.gpkg', 'nc_rail_and_bus_access.gpkg',
            'nc_airport_candidates.gpkg', 'nc_housing_epa_integrated.gpkg']
if REBUILD_ZILLOW_EPA:
    require_files([DATA_DIR / 'zillowzipcode.csv', DATA_DIR / 'SmartLocationDatabase.gdb'], 'Zillow/EPA rebuild needs raw inputs.')
else:
    require_files([PROCESSED_DIR / f for f in expected[1:4]], 'Copy the shared processed housing CSVs, or enable their source rebuild.')
if REBUILD_AMENITIES:
    require_files([OSM_PATH, PADUS_PATH], 'Amenity rebuild needs raw geographic files.')
else:
    require_files([PROCESSED_DIR / f for f in expected[4:9]], 'Copy the shared processed amenity files, or enable REBUILD_AMENITIES.')
if not REBUILD_ACS:
    require_files([PROCESSED_DIR / expected[0]], 'Copy the processed ACS CSV, or enable REBUILD_ACS.')
if not REBUILD_AREAS:
    require_files([PROCESSED_DIR / expected[-1]], 'Copy the saved integrated area file, or set REBUILD_AREAS=True.')
display(pd.DataFrame({'processed_file': expected, 'present': [(PROCESSED_DIR / f).exists() for f in expected]}))

,processed_file,present
0,nc_acs_2024_tract_clean.csv,True
1,nc_zillow_zip_clean.csv,True
2,nc_epa_2010_tract_clean.csv,True
3,nc_zillow_epa_zip_clean.csv,True
4,osm_amenities_clean.gpkg,True
5,padus_open_clean.gpkg,True
6,nc_coastal_access_clean.gpkg,True
7,nc_rail_and_bus_access.gpkg,True
8,nc_airport_candidates.gpkg,True
9,nc_housing_epa_integrated.gpkg,True


## 2. ACS extraction and cleaning

Source: `acs_data_cleaning.ipynb`. Set `REBUILD_ACS=True` to execute this section. It preserves the original nationwide extraction and NC-only export. Otherwise the processed NC table is loaded in Section 5.

The five-year estimates cover 2020-2024. Negative missing-value codes become missing, tract GEOIDs stay as text, and reported medians remain medians for the original source tract. `rent_pct_income` is the median gross-rent-to-income percentage, not the percentage of households that are rent burdened.

In [73]:
if REBUILD_ACS:
    # ── Census API key ────────────────────────────────────────────────────────────
    CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "")

    # ── Dataset ───────────────────────────────────────────────────────────────────
    YEAR = 2024                    # ACS 5-year release: 2020–2024 estimates
    BASE_URL = f"https://api.census.gov/data/{YEAR}/acs/acs5"   # Detailed Tables endpoint

    # Variables pulled for every tract: {Census variable ID: friendly column name}.
    # Add more as the project grows (the API allows up to 50 per request).
    # Browse variables: https://api.census.gov/data/2024/acs/acs5/variables.html
    VARIABLES = {
        # Mapped measures
        "B19013_001E": "median_hh_income",              # Median household income (2024 dollars)
        "B25064_001E": "median_gross_rent",             # Median gross rent (contract rent + utilities)
        "B25071_001E": "rent_pct_income",               # Median gross rent as % of household income
        "B25077_001E": "median_home_value",             # Median value, owner-occupied units
        "B25105_001E": "median_monthly_housing_cost",   # Median monthly housing costs, all occupied units
        # Supporting fields (counts used to weight averages, plus a margin of error for quality checks)
        "B19013_001M": "median_hh_income_moe",
        "B01003_001E": "total_population",
        "B25003_001E": "occupied_units",                # = households
        "B25003_002E": "owner_units",
        "B25003_003E": "renter_units",
    }

    INCLUDE_PUERTO_RICO = False    # ACS 5-year covers Puerto Rico; set True to add it as a map inset
    NC_FIPS = "37"                 # North Carolina

    # ── Folders ───────────────────────────────────────────────────────────────────
    ACS_CACHE = DATA_DIR / f"acs5_{YEAR}_tracts_raw.csv"

In [74]:
if REBUILD_ACS:
    def _check_key():
        if not CENSUS_API_KEY or CENSUS_API_KEY == "YOUR_API_KEY_HERE":
            raise ValueError(
                "Set CENSUS_API_KEY in the configuration cell (or as an environment variable) "
                "before fetching data. Request a free key at https://api.census.gov/data/key_signup.html"
            )


    def census_get(params, retries=4):
        # Call the ACS API and return the JSON rows (first row is the header).
        params = list(params) + [("key", CENSUS_API_KEY)]
        for attempt in range(retries):
            wait = 2 ** attempt * 2
            try:
                r = requests.get(BASE_URL, params=params, timeout=90)
            except requests.RequestException as e:
                print(f"  network error ({type(e).__name__}); retrying in {wait}s")
                time.sleep(wait)
                continue
            if r.status_code == 200:
                try:
                    return r.json()
                except ValueError:
                    # An invalid or not-yet-activated key returns an HTML page instead of JSON.
                    raise RuntimeError(
                        "The Census API did not return JSON. This usually means the API key is invalid "
                        "or hasn't been activated yet (check your email for the activation link).\n"
                        f"Response begins: {r.text[:200]!r}"
                    )
            if r.status_code in (429, 500, 502, 503, 504):
                print(f"  HTTP {r.status_code}; retrying in {wait}s")
                time.sleep(wait)
                continue
            raise RuntimeError(f"Census API error {r.status_code}: {r.text[:300]}")
        raise RuntimeError("Census API request failed after several retries.")


    def list_states():
        rows = census_get([("get", "NAME"), ("for", "state:*")])
        states = pd.DataFrame(rows[1:], columns=rows[0])
        if not INCLUDE_PUERTO_RICO:
            states = states[states["state"] != "72"]
        return states.sort_values("state").reset_index(drop=True)


    def fetch_state_tracts(state_fips):
        rows = census_get([
            ("get", ",".join(["NAME", *VARIABLES])),
            ("for", "tract:*"),
            ("in", f"state:{state_fips}"),
            ("in", "county:*"),
        ])
        return pd.DataFrame(rows[1:], columns=rows[0])


    def load_or_fetch_acs():
        if ACS_CACHE.exists():
            cached = pd.read_csv(ACS_CACHE, dtype=str)   # dtype=str keeps leading zeros in FIPS codes
            if set(VARIABLES).issubset(cached.columns):
                print(f"Loaded {len(cached):,} tracts from cache ({ACS_CACHE})")
                return cached
            print("Cached file is missing some variables; fetching fresh data.")

        _check_key()
        states = list_states()
        frames = []
        for i, row in states.iterrows():
            print(f"[{i + 1:2d}/{len(states)}] {row['NAME']:<22}", end="")
            df = fetch_state_tracts(row["state"])
            print(f"{len(df):>6,} tracts")
            frames.append(df)
            time.sleep(0.2)   # be polite to the API
        raw = pd.concat(frames, ignore_index=True)
        raw.to_csv(ACS_CACHE, index=False)
        print(f"\nSaved {len(raw):,} tracts to {ACS_CACHE}")
        return raw


    raw = load_or_fetch_acs()
    display(raw.head())

In [75]:
if REBUILD_ACS:
    acs = raw.copy()
    acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"]

    for var_id, name in VARIABLES.items():
        values = pd.to_numeric(acs[var_id], errors="coerce")
        acs[name] = values.mask(values < 0)    # ACS "not available" codes are all negative

    acs = acs[["GEOID", "NAME", "state", "county", "tract", *VARIABLES.values()]]
    acs.to_csv(DATA_DIR / f"acs5_{YEAR}_tracts_clean.csv", index=False)

    print(f"{len(acs):,} tracts")
    display(acs.describe().round(1))

### Optional ACS maps and distributions

The original nationwide and NC choropleths and distributions are retained below. Enable both `REBUILD_ACS` and `DRAW_SOURCE_MAPS` to generate them. A household-weighted average of tract medians is a descriptive summary, not a pooled household median.

In [76]:
if REBUILD_ACS and DRAW_SOURCE_MAPS:
    def money_short(v):        # $35k, $1.5M
        if v <= 0:
            return "$0"
        return f"${v / 1e6:g}M" if v >= 1e6 else f"${v / 1000:,.0f}k"


    def money_full(v):         # $1,200
        return f"${v:,.0f}"


    def pct(v):                # 30%
        return f"{v:.0f}%"


    MAP_SPECS = {
        "median_hh_income": dict(
            title="Median Household Income",
            label="Median household income (2024 dollars)",
            table="B19013",
            breaks=[0, 35_000, 50_000, 65_000, 80_000, 100_000, 125_000, 150_000, 200_000, 250_000],
            fmt=money_short, top_label="$250k+",
            cmap="magma", cmap_range=(0.08, 0.97),
            weight="occupied_units",
        ),
        "median_gross_rent": dict(
            title="Median Gross Rent",
            label="Median monthly gross rent, including utilities (2024 dollars)",
            table="B25064",
            breaks=[0, 800, 1_000, 1_200, 1_400, 1_600, 1_900, 2_200, 2_700, 3_500],
            fmt=money_full, top_label="$3,500+",
            cmap="viridis", cmap_range=(0.0, 1.0),
            weight="renter_units",
        ),

        # "rent_pct_income": dict(
        #     title="Rent as a Share of Household Income",
        #     label="Median gross rent as % of household income (30%+ = cost-burdened)",
        #     table="B25071",
        #     breaks=[0, 20, 24, 27, 30, 33, 37, 42, 50],
        #     fmt=pct, top_label="50%+",
        #     cmap="RdYlBu_r", cmap_range=(0.0, 1.0),
        #     outline="#555555",      # white borders disappear against the pale middle colors
        #     weight="renter_units",
        # ),
            "rent_pct_income": dict(
            title="Rent as a Share of Household Income",
            label="Median gross rent as % of household income (30%+ = cost-burdened)",
            table="B25071",
            breaks=[0, 20, 25, 30, 35, 40, 50],
            fmt=pct, top_label="50%+",
            # Hand-picked from ColorBrewer RdBu with the pale center shades removed:
            # 3 blues below the 30% threshold, 3 reds above it, so every bin is distinct.
            colors=["#2166ac", "#4393c3", "#92c5de", "#f4a582", "#d6604d", "#b2182b"],
            outline="#555555",
            weight="renter_units",
        ),

        "median_home_value": dict(
            title="Median Home Value",
            label="Median value of owner-occupied homes (2024 dollars)",
            table="B25077",
            breaks=[0, 100_000, 150_000, 200_000, 250_000, 325_000, 400_000, 500_000, 750_000, 1_000_000, 2_000_000],
            fmt=money_short, top_label="$2M+",
            cmap="plasma", cmap_range=(0.0, 0.95),
            weight="owner_units",
        ),

        "median_monthly_housing_cost": dict(
            title="Median Monthly Housing Costs",
            label="Median monthly housing costs, owners and renters (2024 dollars)",
            table="B25105",
            breaks=[0, 600, 800, 1_000, 1_200, 1_500, 1_800, 2_200, 2_800, 4_000],
            fmt=money_full, top_label="$4,000+",
            cmap="YlGnBu", cmap_range=(0.12, 1.0),
            outline="#555555",
            weight="occupied_units",
        ),
    }

    # Draw all of them, or list just the ones you want, e.g. ["median_gross_rent"]
    MAPS_TO_DRAW = list(MAP_SPECS)

In [77]:
if REBUILD_ACS and DRAW_SOURCE_MAPS:
    def describe_measure(df, col, weight_col):
        v = df[col]
        ok = v.notna() & (df[weight_col] > 0)
        return {
            "Tracts with data": v.notna().sum(),
            "Median of tracts": v.median(),
            "Mean of tracts": v.mean(),
            "Household-weighted mean": np.average(v[ok], weights=df.loc[ok, weight_col]) if ok.any() else np.nan,
            "10th percentile": v.quantile(0.10),
            "90th percentile": v.quantile(0.90),
        }


    rows = []
    nc_acs = acs[acs["state"] == NC_FIPS]
    for col, spec in MAP_SPECS.items():
        for place, df in [("United States", acs), ("North Carolina", nc_acs)]:
            rows.append({"Measure": spec["title"], "Area": place, **describe_measure(df, col, spec["weight"])})

    summary = pd.DataFrame(rows).set_index(["Measure", "Area"])

    # Display: whole numbers, except one decimal for the rent-burden percentages
    shown = summary.apply(lambda c: c.map(lambda v: f"{v:,.0f}"))
    pct_rows = summary.index.get_level_values("Measure") == MAP_SPECS["rent_pct_income"]["title"]
    stat_cols = summary.columns[1:]
    shown.loc[pct_rows, stat_cols] = summary.loc[pct_rows, stat_cols].apply(lambda c: c.map(lambda v: f"{v:.1f}%"))
    display(shown)

In [78]:
if REBUILD_ACS and DRAW_SOURCE_MAPS:
    GEO_BASE = f"https://www2.census.gov/geo/tiger/GENZ{YEAR}/shp"


    def get_boundaries(layer):
        # layer is "tract", "county", or "state"
        fname = f"cb_{YEAR}_us_{layer}_500k.zip"
        path = DATA_DIR / fname
        if not path.exists():
            print(f"Downloading {fname} ...")
            with requests.get(f"{GEO_BASE}/{fname}", stream=True, timeout=600) as r:
                r.raise_for_status()
                with open(path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
        return gpd.read_file(path)


    keep_states = set(acs["state"])     # the 50 states + DC (+ PR if enabled)

    tracts_geo = get_boundaries("tract")
    tracts_geo = tracts_geo[tracts_geo["STATEFP"].isin(keep_states)]
    counties_geo = get_boundaries("county")
    counties_geo = counties_geo[counties_geo["STATEFP"].isin(keep_states)]
    states_geo = get_boundaries("state")
    states_geo = states_geo[states_geo["STATEFP"].isin(keep_states)]

    # Join ACS values onto tract shapes; drop water-only tracts (no land, no residents)
    gdf = tracts_geo[["GEOID", "STATEFP", "ALAND", "geometry"]].merge(acs, on="GEOID", how="left")
    gdf = gdf[pd.to_numeric(gdf["ALAND"]) > 0]

    unmatched = len(set(acs["GEOID"]) - set(tracts_geo["GEOID"]))
    print(f"{len(gdf):,} tract shapes; ACS tracts without a matching shape: {unmatched}")
    print((gdf[list(MAP_SPECS)].notna().sum()).rename("tracts with data").to_frame())

In [79]:
if REBUILD_ACS and DRAW_SOURCE_MAPS:
    CONUS_CRS, ALASKA_CRS, HAWAII_CRS, PR_CRS, NC_CRS = "EPSG:5070", "EPSG:3338", "ESRI:102007", "EPSG:32161", "EPSG:32119"
    OUTSIDE_CONUS = ["02", "15", "72"]
    NO_DATA_COLOR = "#cfcfcf"
    SOURCE_NOTE = (f"Source: U.S. Census Bureau, American Community Survey {YEAR - 4}–{YEAR} 5-Year Estimates "
                   f"(Table {{table}}); {YEAR} Cartographic Boundary Files.")

    # National layers
    lower48 = gdf[~gdf["STATEFP"].isin(OUTSIDE_CONUS)].to_crs(CONUS_CRS)
    states48 = states_geo[~states_geo["STATEFP"].isin(OUTSIDE_CONUS)].to_crs(CONUS_CRS)

    inset_specs = [("02", ALASKA_CRS, [0.02, 0.11, 0.24, 0.24]),
                   ("15", HAWAII_CRS, [0.25, 0.13, 0.13, 0.11])]
    if INCLUDE_PUERTO_RICO:
        inset_specs.append(("72", PR_CRS, [0.39, 0.13, 0.08, 0.06]))
    insets = [(gdf[gdf["STATEFP"] == fips].to_crs(crs),
               states_geo[states_geo["STATEFP"] == fips].to_crs(crs), pos)
              for fips, crs, pos in inset_specs]

    # North Carolina layers
    nc = gdf[gdf["STATEFP"] == NC_FIPS].to_crs(NC_CRS)
    nc_counties = counties_geo[counties_geo["STATEFP"] == NC_FIPS].to_crs(NC_CRS)
    nc_outline = states_geo[states_geo["STATEFP"] == NC_FIPS].to_crs(NC_CRS)

    # "side" sets which side of the dot each label sits on, to keep nearby labels from colliding
    cities = pd.DataFrame({
        "name": ["Charlotte", "Raleigh", "Greensboro", "Durham", "Winston-Salem",
                 "Fayetteville", "Wilmington", "Asheville"],
        "lat": [35.2271, 35.7796, 36.0726, 35.9940, 36.0999, 35.0527, 34.2104, 35.5951],
        "lon": [-80.8431, -78.6382, -79.7920, -78.8986, -80.2442, -78.8784, -77.8868, -82.5515],
        "side": ["right", "right", "right", "right", "left", "right", "right", "right"],
    })
    cities = gpd.GeoDataFrame(cities, geometry=gpd.points_from_xy(cities.lon, cities.lat),
                              crs="EPSG:4326").to_crs(NC_CRS)


    # def scale_for(spec):
    #     # Discrete colormap + norm for a measure's bins
    #     lo, hi = spec["cmap_range"]
    #     n_bins = len(spec["breaks"]) - 1
    #     cmap = ListedColormap(plt.get_cmap(spec["cmap"])(np.linspace(lo, hi, n_bins)))
    #     return cmap, BoundaryNorm(spec["breaks"], n_bins)

    def scale_for(spec):
        # Discrete colormap + norm for a measure's bins
        n_bins = len(spec["breaks"]) - 1
        if "colors" in spec:                      # explicit list of colors, one per bin
            assert len(spec["colors"]) == n_bins, f"{spec['title']}: need {n_bins} colors"
            cmap = ListedColormap(spec["colors"])
        else:                                     # sample a named matplotlib colormap
            lo, hi = spec["cmap_range"]
            cmap = ListedColormap(plt.get_cmap(spec["cmap"])(np.linspace(lo, hi, n_bins)))
        return cmap, BoundaryNorm(spec["breaks"], n_bins)

    def plot_tracts(ax, data, col, spec):
        # Fill tracts by bin; values above the last cutoff (incl. top-codes) land in the top bin.
        cmap, norm = scale_for(spec)
        breaks = spec["breaks"]
        has = data[col].notna()
        if has.any():
            vals = data.loc[has, col].clip(lower=breaks[0], upper=breaks[-1] - 1e-6).to_numpy()
            data[has].plot(ax=ax, color=cmap(norm(vals)), linewidth=0)
        if (~has).any():
            data[~has].plot(ax=ax, color=NO_DATA_COLOR, linewidth=0)
        ax.set_axis_off()


    def add_legend(fig, cax, spec):
        cmap, norm = scale_for(spec)
        breaks = spec["breaks"]
        cb = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation="horizontal",
                          spacing="uniform", ticks=breaks)
        cb.ax.set_xticklabels([spec["fmt"](b) for b in breaks[:-1]] + [spec["top_label"]], fontsize=10)
        cb.outline.set_visible(False)
        cb.ax.tick_params(length=0, pad=4)
        cb.set_label(spec["label"], fontsize=12, labelpad=8)
        cax.legend(handles=[Patch(color=NO_DATA_COLOR, label="No data")], loc="center left",
                   bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10, handlelength=1.2)

In [80]:
if REBUILD_ACS and DRAW_SOURCE_MAPS:
    def draw_us_map(col, show=True):
        spec = MAP_SPECS[col]
        fig = plt.figure(figsize=(16, 11))
        ax = fig.add_axes([0.01, 0.12, 0.98, 0.78])
        plot_tracts(ax, lower48, col, spec)
        states48.boundary.plot(ax=ax, color=spec.get("outline", "white"), linewidth=0.5)

        for tracts_inset, outline_inset, pos in insets:
            inset_ax = fig.add_axes(pos)
            plot_tracts(inset_ax, tracts_inset, col, spec)
            outline_inset.boundary.plot(ax=inset_ax, color="#555555", linewidth=0.4)

        fig.text(0.5, 0.955, f"{spec['title']} by Census Tract", ha="center", fontsize=24, weight="bold")
        fig.text(0.5, 0.925, f"United States, ACS {YEAR - 4}–{YEAR} 5-Year Estimates",
                 ha="center", fontsize=14, color="#444444")
        add_legend(fig, fig.add_axes([0.50, 0.08, 0.40, 0.022]), spec)
        fig.text(0.99, 0.015, SOURCE_NOTE.format(table=spec["table"]), ha="right", fontsize=9, color="#666666")

        path = FIG_DIR / f"us_tract_{col}_{YEAR}.png"
        fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"Saved {path}")
        if show:
            plt.show()
        plt.close(fig)


    def draw_nc_map(col, show=True):
        spec = MAP_SPECS[col]
        fig, ax = plt.subplots(figsize=(16, 7))
        plot_tracts(ax, nc, col, spec)
        nc_counties.boundary.plot(ax=ax, color=spec.get("outline", "white"), linewidth=0.4, alpha=0.8)
        nc_outline.boundary.plot(ax=ax, color="#333333", linewidth=0.9)

        halo = [pe.withStroke(linewidth=3, foreground="white")]
        for _, c in cities.iterrows():
            left = c["side"] == "left"
            ax.plot(c.geometry.x, c.geometry.y, "o", ms=5, mfc="white", mec="black", mew=1, zorder=5)
            ax.annotate(c["name"], (c.geometry.x, c.geometry.y), xytext=(-6 if left else 6, 5),
                        textcoords="offset points", ha="right" if left else "left",
                        fontsize=11, weight="bold", path_effects=halo, zorder=6)

        ax.set_title(f"{spec['title']} by Census Tract — North Carolina\n"
                     f"ACS {YEAR - 4}–{YEAR} 5-Year Estimates", fontsize=18, weight="bold", pad=14)
        add_legend(fig, ax.inset_axes([0.03, 0.06, 0.46, 0.04]), spec)
        fig.text(0.99, 0.01, SOURCE_NOTE.format(table=spec["table"]), ha="right", fontsize=9, color="#666666")

        path = FIG_DIR / f"nc_tract_{col}_{YEAR}.png"
        fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"Saved {path}")
        if show:
            plt.show()
        plt.close(fig)

In [81]:
if REBUILD_ACS and DRAW_SOURCE_MAPS:
    for col in MAPS_TO_DRAW:
        draw_us_map(col)

In [82]:
if REBUILD_ACS and DRAW_SOURCE_MAPS:
    for col in MAPS_TO_DRAW:
        draw_nc_map(col)

In [83]:
if REBUILD_ACS and DRAW_SOURCE_MAPS:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
    for ax, (col, spec) in zip(axes.flat, MAP_SPECS.items()):
        top = spec["breaks"][-1]
        bins = np.linspace(0, top, 31)
        us_vals = acs[col].dropna().clip(upper=top)
        nc_vals = nc_acs[col].dropna().clip(upper=top)
        ax.hist(us_vals, bins=bins, density=True, alpha=0.55, color="#3b528b", label="United States")
        ax.hist(nc_vals, bins=bins, density=True, histtype="step", linewidth=2, color="#e16462",
                label="North Carolina")
        ymax = ax.get_ylim()[1]
        for name, vals, color, y in [("US", us_vals, "#3b528b", 0.93), ("NC", nc_vals, "#e16462", 0.80)]:
            med = vals.median()
            ax.axvline(med, color=color, linestyle="--", linewidth=1.2)
            ax.text(med, ymax * y, f" {name} {spec['fmt'](med)}", color=color, fontsize=10)
        # default args bind this panel's spec (a plain closure would use the last spec in the loop)
        ax.xaxis.set_major_formatter(lambda v, _, s=spec, t=top: s["top_label"] if v >= t else s["fmt"](v))
        ax.set_yticks([])
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.set_title(spec["title"], fontsize=12, weight="bold", loc="left")

    legend_ax = axes.flat[-1]
    legend_ax.axis("off")
    legend_ax.legend(*axes.flat[0].get_legend_handles_labels(), loc="center", frameon=False, fontsize=13,
                     title="Distribution of tract medians\n(dashed line = median tract)", title_fontsize=12)
    fig.suptitle(f"How North Carolina's Census Tracts Compare to the Nation (ACS {YEAR - 4}–{YEAR})",
                 fontsize=16, weight="bold")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"distributions_us_vs_nc_{YEAR}.png", dpi=300, bbox_inches="tight", facecolor="white")
    display(plt.show())

In [84]:
if REBUILD_ACS:
    from pathlib import Path

    export_dir = PROCESSED_DIR
    export_dir.mkdir(parents=True, exist_ok=True)

    acs.loc[acs["state"].eq("37")].to_csv(
        export_dir / "nc_acs_2024_tract_clean.csv",
        index=False
    )

    print("Saved NC ACS data.")

## 3. Zillow and EPA preparation

Source: `zillow_epa_data_cleaning.ipynb`. Enable `REBUILD_ZILLOW_EPA` to run this section.

Zillow `home_value` is the latest dated value in the supplied file, while `home_value_12mo` is the mean of up to the final 12 monthly columns. This is a home-value index, not a rental price or observed sale price.

EPA block groups are aggregated to 2010 tracts using population weights. The original population-weighted ZIP version is retained as a secondary export. The final area dataset later uses the **tract-level** EPA table, assigned by geographic overlap.

The original `has_transit` definition uses nonmissing `D4A` as a proxy. Confirm the EPA field definitions and sentinel treatment before interpreting it as confirmed transit service. Aggregated values are population-weighted shares, not binary tract indicators.

In [85]:
if REBUILD_ZILLOW_EPA:
    zillowzipcode = pd.read_csv(DATA_DIR / "zillowzipcode.csv", dtype={"RegionName": str})

In [86]:
if REBUILD_ZILLOW_EPA:
    # Make sure every ZIP is 5 digits (safety check; should already be true with dtype=str)
    zillowzipcode['RegionName'] = zillowzipcode['RegionName'].str.zfill(5)

    # Month columns are the ones whose names start with a year, e.g. '2026-09-30'
    date_cols = sorted([c for c in zillowzipcode.columns if pd.notna(pd.to_datetime(c, format='%Y-%m-%d', errors='coerce'))])
    if not date_cols:
        raise ValueError('Zillow file has no dated monthly columns.')
    zillowzipcode[date_cols] = zillowzipcode[date_cols].apply(pd.to_numeric, errors='coerce')
    latest = date_cols[-1]

    # Build a smaller table: location info + latest value + 12-month average
    zillow = zillowzipcode[['RegionName', 'State', 'City', 'Metro', 'CountyName']].copy()
    zillow['home_value'] = zillowzipcode[latest]
    zillow['home_value_12mo'] = zillowzipcode[date_cols[-12:]].mean(axis=1)

    # Rename RegionName to zip, and drop any ZIP without a current value
    zillow = zillow.rename(columns={'RegionName': 'zip'}).dropna(subset=['home_value'])

    print(zillow.shape)
    display(zillow.head())

In [87]:
if REBUILD_ZILLOW_EPA:
    # Check missing values: count and percent per column
    missing = pd.DataFrame({
        'missing': zillow.isna().sum(),
        'percent': (zillow.isna().mean() * 100).round(1)
    })
    display(missing[missing['missing'] > 0])

In [88]:
if REBUILD_ZILLOW_EPA:
    # Fill missing categories with a label instead of dropping rows.
    # Metro is blank for ~4,709 rural ZIPs that aren't in any metro area. That's real
    # information, not bad data, so we label them 'Non-metro' and keep them.
    # City is blank for 7 ZIPs.
    # fillna only fills empty cells; existing values are untouched.
    # We do NOT fill numeric columns like home_value. Made-up prices would distort the model.
    zillow['Metro'] = zillow['Metro'].fillna('Non-metro')
    zillow['City'] = zillow['City'].fillna('Unknown')

    # Verify: should print 0 for every column
    display(zillow.isna().sum())

In [89]:
if REBUILD_ZILLOW_EPA:
    #IMPORTING SmartLocationDatabase.gdb
    # EPA Smart Location Database: built-environment features per Census block group (~2018 data).
    # read_geometry=False skips the map shapes, which we don't need, so loading is much faster.
    sld_cols = ['GEOID10', 'STATEFP', 'TotPop', 'HH', 'Ac_Land', 'TotEmp',
                'D1A', 'D1C', 'D2B_E8MIXA', 'D3B', 'D4A', 'D5AR', 'NatWalkInd',
                'Pct_AO0', 'R_PCTLOWWAGE', 'VMT_per_worker']
    sld = pyogrio.read_dataframe(DATA_DIR / "SmartLocationDatabase.gdb",
                                 columns=sld_cols, read_geometry=False)

    # EPA codes "no value" as -99999 (mostly transit columns), so turn it into real NaN
    sld = sld.replace(-99999, np.nan)

    # Drop territories (Zillow doesn't cover them) and block groups with nobody living there
    sld = sld[~sld['STATEFP'].isin(['60', '66', '69', '72', '78'])]
    sld = sld[sld['TotPop'] > 0]

    # 1 if there's a transit stop within walking distance, 0 if not
    sld['has_transit'] = sld['D4A'].notna().astype(int)

    # First 11 digits of a block group ID = its tract ID
    sld['tract'] = sld['GEOID10'].astype('string').str[:11]

In [90]:
if REBUILD_ZILLOW_EPA:
    # Census crosswalk: which tracts overlap each ZIP, and how many people live in each overlapping piece.
    # (Census calls them ZCTAs, which are their approximation of ZIP codes. 2010 boundaries, matching the SLD.)
    # dtype=str keeps leading zeros on both IDs.
    xw_url = 'https://www2.census.gov/geo/docs/maps-data/data/rel/zcta_tract_rel_10.txt'
    xw = pd.read_csv(download_cached(xw_url, DATA_DIR / 'raw' / 'zcta_tract_rel_10.txt'), usecols=['ZCTA5', 'GEOID', 'POPPT'], dtype={'ZCTA5': str, 'GEOID': str})
    xw = xw.rename(columns={'ZCTA5': 'zip', 'GEOID': 'tract', 'POPPT': 'pop_in_zip'})

    # Drop overlaps where nobody lives (e.g. a ZIP clipping an empty corner of a tract)
    xw = xw[xw['pop_in_zip'] > 0]

In [91]:
if REBUILD_ZILLOW_EPA:
    # Features to average for each ZIP
    rate_cols = ['D1A', 'D1C', 'D2B_E8MIXA', 'D3B', 'D5AR', 'NatWalkInd',
                 'Pct_AO0', 'R_PCTLOWWAGE', 'VMT_per_worker', 'has_transit']

    # Population-weighted average of rate_cols within each group.
    # Missing values are skipped (their weight isn't counted), so they don't drag averages toward 0.
    def weighted_mean(df, group_col, weight_col):
        num = df[rate_cols].mul(df[weight_col], axis=0).groupby(df[group_col]).sum()
        den = df[rate_cols].notna().mul(df[weight_col], axis=0).groupby(df[group_col]).sum()
        return (num / den).reset_index()

    # Step 1: block groups -> tracts, weighted by each block group's population
    sld_by_tract = weighted_mean(sld, 'tract', 'TotPop')

    # Step 2: tracts -> ZIPs, weighted by how many people live in each tract/ZIP overlap
    xw_sld = xw.merge(sld_by_tract, on='tract', how='inner')
    sld_zip = weighted_mean(xw_sld, 'zip', 'pop_in_zip')

    print(sld_by_tract.shape)  # expect (72411, 11)
    print(sld_zip.shape)       # expect (32843, 11)

In [92]:
if REBUILD_ZILLOW_EPA:
    housing = zillow.merge(sld_zip, on='zip', how='inner', validate='one_to_one')

    print(housing.shape)              # expect (26208, 17)
    print(housing.isna().sum().sum()) # expect 0
    display(housing.head())

In [93]:
if REBUILD_ZILLOW_EPA:
    # Which Zillow ZIPs didn't match?
    unmatched = zillow[~zillow['zip'].isin(sld_zip['zip'])]
    print(len(unmatched), 'unmatched')
    display(unmatched.head(10))

In [94]:
if REBUILD_ZILLOW_EPA:
    from pathlib import Path

    export_dir = PROCESSED_DIR
    export_dir.mkdir(parents=True, exist_ok=True)

    # Zillow values by ZIP
    zillow.loc[zillow["State"].eq("NC")].to_csv(
        export_dir / "nc_zillow_zip_clean.csv",
        index=False
    )

    # EPA features by 2010 census tract
    sld_by_tract.loc[
        sld_by_tract["tract"].str.startswith("37")
    ].to_csv(
        export_dir / "nc_epa_2010_tract_clean.csv",
        index=False
    )

    # Existing Zillow + EPA merge by ZIP
    housing.loc[housing["State"].eq("NC")].to_csv(
        export_dir / "nc_zillow_epa_zip_clean.csv",
        index=False
    )

    print("Saved three NC datasets.")

## 4. Amenities and access preparation

Source: `amenities_and_accessibility.ipynb`. Enable `REBUILD_AMENITIES` to run this section.

OSM features are grouped into food/nightlife, shopping/daily needs, sports/fitness, entertainment/culture, and outdoor recreation. Point/polygon duplicates are consolidated only for unambiguous same-name, same-category intersecting pairs. PAD-US records are selected using open-access classifications.

Coastal and rail APIs are retrieved in batches with ID-completeness checks. Airport matches remain candidates. Piedmont Triad was missing in the original snapshot; current matching results are displayed rather than assumed to remain unchanged. These classifications do not verify current schedules or public access.

In [95]:
if REBUILD_AMENITIES:
    # Lists layers without loading all their geographic features
    osm_layers = pd.DataFrame(
        pyogrio.list_layers(OSM_PATH),
        columns=["layer", "geometry_type"]
    )

    padus_layers = pd.DataFrame(
        pyogrio.list_layers(PADUS_PATH),
        columns=["layer", "geometry_type"]
    )

    print("OpenStreetMap layers")
    display(osm_layers)

    print("PAD-US layers")
    display(padus_layers)

In [96]:
if REBUILD_AMENITIES:
    # Businesses and amenities can be mapped as points or area outlines
    osm_poi_points = gpd.read_file(
        OSM_PATH,
        layer="gis_osm_pois_free",
        engine="pyogrio"
    )

    osm_poi_areas = gpd.read_file(
        OSM_PATH,
        layer="gis_osm_pois_a_free",
        engine="pyogrio"
    )

    for label, df in [
        ("POI points", osm_poi_points),
        ("POI areas", osm_poi_areas)
    ]:
        print(f"\n{label}: {len(df):,} records")
        print("CRS:", df.crs)
        print("Columns:", df.columns.tolist())
        display(df.head(3))

In [97]:
if REBUILD_AMENITIES:
    category_counts = pd.concat(
        [
            osm_poi_points["fclass"].value_counts().rename("points"),
            osm_poi_areas["fclass"].value_counts().rename("areas")
        ],
        axis=1
    ).fillna(0).astype(int)

    category_counts["total_records"] = category_counts.sum(axis=1)

    # These are raw record counts, not yet deduplicated venue counts
    display(
        category_counts.sort_values(
            "total_records", ascending=False
        ).rename_axis("category")
    )

In [98]:
if REBUILD_AMENITIES:
    PADUS_LAYER = "PADUS4_1Comb_DOD_Trib_NGP_Fee_Desig_Ease_State_NC"

    # Read a small sample first
    padus_sample = gpd.read_file(
        PADUS_PATH,
        layer=PADUS_LAYER,
        rows=5,
        engine="pyogrio"
    )

    print("PAD-US columns:")
    print(padus_sample.columns.tolist())
    display(padus_sample)

    # Read the lookup table that explains public-access codes
    public_access_lookup = pyogrio.read_dataframe(
        PADUS_PATH,
        layer="Public_Access",
        read_geometry=False
    )

    print("\nPublic-access definitions:")
    display(public_access_lookup)

In [99]:
if REBUILD_AMENITIES:
    padus = gpd.read_file(
        PADUS_PATH,
        layer=PADUS_LAYER,
        engine="pyogrio"
    )

    print(f"PAD-US records: {len(padus):,}")
    print("CRS:", padus.crs)

    display(
        padus.groupby(
            ["Pub_Access", "d_Pub_Access"],
            dropna=False
        ).size().rename("records").reset_index()
    )

In [100]:
if REBUILD_AMENITIES:
    # Keep restricted-access records separately for possible later use
    padus_open = padus[
        padus["Pub_Access"].eq("OA")
    ].copy()

    padus_restricted = padus[
        padus["Pub_Access"].eq("RA")
    ].copy()

    # Remove missing or empty geometries
    padus_open = padus_open[
        padus_open.geometry.notna()
        & ~padus_open.geometry.is_empty
    ].copy()

    # Project to NC State Plane in meters for later distance calculations
    padus_open = padus_open.to_crs("EPSG:32119")

    print(f"Open-access land records: {len(padus_open):,}")

    display(
        padus_open["d_Des_Tp"]
        .value_counts(dropna=False)
        .rename_axis("designation")
        .reset_index(name="records")
    )

    display(
        padus_open[
            ["Unit_Nm", "d_Des_Tp", "d_Pub_Access"]
        ].head(10)
    )

In [101]:
if REBUILD_AMENITIES:
    # Project-defined groups for the recommender
    land_groups = {
        "parks_and_recreation": [
            "Local Park",
            "Local Recreation Area",
            "State Park",
            "State Recreation Area",
            "Recreation Management Area",
            "Access Area"
        ],
        "forests_and_nature": [
            "National Forest",
            "National Park",
            "Inventoried Roadless Area",
            "Local Conservation Area",
            "State Conservation Area",
            "Conservation Area",
            "Private Conservation",
            "National Public Lands",
            "Local Resource Management Area",
            "State Resource Management Area",
            "Watershed Protection Area"
        ],
        "coastal_and_river_lands": [
            "National Lakeshore or Seashore",
            "Wild and Scenic River"
        ],
        "historic_and_cultural": [
            "Local Historic or Cultural Area",
            "State Historic or Cultural Area"
        ]
    }

    designation_to_group = {
        designation: group
        for group, designations in land_groups.items()
        for designation in designations
    }

    padus_open["amenity_group"] = (
        padus_open["d_Des_Tp"]
        .map(designation_to_group)
        .fillna("other_open_land")
    )

    display(
        padus_open.groupby("amenity_group")
        .size()
        .reset_index(name="land_records")
    )

In [102]:
if REBUILD_AMENITIES:
    # Candidate categories to check against your actual OSM data
    candidate_groups = {
        "food_and_nightlife": [
            "restaurant", "fast_food", "cafe", "pub", "bar",
            "biergarten", "nightclub", "food_court"
        ],
        "shopping_and_daily_needs": [
            "supermarket", "convenience", "pharmacy",
            "mall", "department_store", "marketplace"
        ],
        "sports_and_fitness": [
            "sports_centre", "stadium", "pitch", "track",
            "golf_course", "swimming_pool", "fitness_centre"
        ],
        "entertainment_and_culture": [
            "cinema", "theatre", "museum", "arts_centre",
            "theme_park", "zoo", "attraction"
        ],
        "outdoor_recreation": [
            "park", "playground", "picnic_site",
            "viewpoint", "camp_site"
        ]
    }

    # Only display categories that actually exist in your file
    for group, categories in candidate_groups.items():
        available = category_counts.index.intersection(categories)

        print(f"\n{group}")
        print(category_counts.loc[available].to_string())

In [103]:
if REBUILD_AMENITIES:
    # Keep the source layer so we can trace each record
    points = osm_poi_points.copy()
    points["source_layer"] = "gis_osm_pois_free"

    areas = osm_poi_areas.copy()
    areas["source_layer"] = "gis_osm_pois_a_free"

    # Use the same projected CRS for future distance calculations
    points = points.to_crs("EPSG:32119")
    areas = areas.to_crs("EPSG:32119")

    osm_amenities = gpd.GeoDataFrame(
        pd.concat([points, areas], ignore_index=True),
        geometry="geometry",
        crs="EPSG:32119"
    )

    # Assign our project categories
    category_to_group = {
        category: group
        for group, categories in candidate_groups.items()
        for category in categories
    }

    osm_amenities["amenity_group"] = (
        osm_amenities["fclass"].map(category_to_group)
    )

    # Keep selected categories and usable geometry
    osm_amenities = osm_amenities[
        osm_amenities["amenity_group"].notna()
        & osm_amenities.geometry.notna()
        & ~osm_amenities.geometry.is_empty
    ].copy()

    # Clean names without dropping unnamed amenities
    osm_amenities["name"] = (
        osm_amenities["name"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    # Traceable record ID, not yet a unique venue ID
    osm_amenities["record_id"] = (
        osm_amenities["source_layer"]
        + ":"
        + osm_amenities["osm_id"].astype("string")
    )

    print(f"Selected amenity records: {len(osm_amenities):,}")

    display(
        osm_amenities.groupby("amenity_group")
        .size()
        .reset_index(name="records")
    )

In [104]:
if REBUILD_AMENITIES:
    # Exact repeated records within the same source layer
    exact_duplicates = osm_amenities.duplicated(
        subset=["source_layer", "osm_id", "fclass", "geometry"]
    )

    print(f"Exact duplicate records: {exact_duplicates.sum():,}")

    osm_amenities = osm_amenities.loc[~exact_duplicates].copy()

    # Repeated names and categories are candidates for review,
    # not confirmed duplicates. Different branches can share a name.
    named = osm_amenities.dropna(subset=["name"]).copy()
    named["name_normalized"] = named["name"].str.casefold()

    potential_duplicates = named[
        named.duplicated(
            subset=["name_normalized", "fclass"],
            keep=False
        )
    ].sort_values(["name_normalized", "fclass"])

    print(f"Records with repeated name/category: {len(potential_duplicates):,}")

    display(
        potential_duplicates[
            ["name", "fclass", "source_layer", "osm_id"]
        ].head(20)
    )

In [105]:
if REBUILD_AMENITIES:
    # Separate named point and polygon records
    poi_points = named[
        named.geometry.geom_type.eq("Point")
    ].copy()

    poi_areas = named[
        named.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    ].copy()

    # Find points inside or on the boundary of amenity polygons
    matches = gpd.sjoin(
        poi_points[
            ["record_id", "name", "name_normalized", "fclass", "geometry"]
        ],
        poi_areas[
            ["record_id", "name_normalized", "fclass", "geometry"]
        ],
        how="inner",
        predicate="intersects",
        lsuffix="point",
        rsuffix="area"
    )

    # Keep only matching names AND categories
    duplicate_candidates = matches[
        matches["name_normalized_point"].eq(
            matches["name_normalized_area"]
        )
        & matches["fclass_point"].eq(matches["fclass_area"])
    ].copy()

    print(f"Potential point/polygon duplicate pairs: {len(duplicate_candidates):,}")

    display(
        duplicate_candidates[
            ["name", "fclass_point", "record_id_point", "record_id_area"]
        ].head(20)
    )

In [106]:
if REBUILD_AMENITIES:
    # Count how many different features match each point and polygon
    pairs = duplicate_candidates[
        ["record_id_point", "record_id_area"]
    ].drop_duplicates()

    point_matches = pairs.groupby("record_id_point")[
        "record_id_area"
    ].nunique()

    area_matches = pairs.groupby("record_id_area")[
        "record_id_point"
    ].nunique()

    # Only consolidate unambiguous one-to-one matches
    clear_pairs = pairs[
        pairs["record_id_point"].map(point_matches).eq(1)
        & pairs["record_id_area"].map(area_matches).eq(1)
    ].copy()

    # Keep an audit table of the point records excluded
    osm_duplicate_audit = clear_pairs.copy()
    osm_duplicate_audit["reason"] = (
        "Same name and category; point intersects polygon; one-to-one match"
    )

    # Create a separate cleaned table, retaining the polygon representation
    osm_amenities_clean = osm_amenities[
        ~osm_amenities["record_id"].isin(clear_pairs["record_id_point"])
    ].copy()

    print(f"Point records excluded: {len(clear_pairs):,}")
    print(f"Remaining amenity records: {len(osm_amenities_clean):,}")
    print(f"Ambiguous pairs retained: {len(pairs) - len(clear_pairs):,}")

    display(
        osm_amenities_clean.groupby("amenity_group")
        .size()
        .reset_index(name="records")
    )

In [107]:
if REBUILD_AMENITIES:
    OUTPUT_DIR = PROCESSED_DIR
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    osm_amenities_clean.to_file(
        OUTPUT_DIR / "osm_amenities_clean.gpkg",
        layer="amenities",
        driver="GPKG",
        engine="pyogrio"
    )

    osm_duplicate_audit.to_csv(
        OUTPUT_DIR / "osm_duplicate_audit.csv",
        index=False
    )

    print("Saved cleaned OSM amenities and duplicate audit.")

In [108]:
if REBUILD_AMENITIES:
    BEACH_URL = (
        "https://services2.arcgis.com/kCu40SDxsCGcuUWO/"
        "ArcGIS/rest/services/DCM_Beach_and_Waterfront_Access/"
        "FeatureServer/0"
    )

    def arcgis_request(url, params):
        response = requests.post(url, data=params, timeout=120)
        response.raise_for_status()
        data = response.json()

        if "error" in data:
            raise RuntimeError(data["error"])

        return data


    # Get IDs first so we can request every record in manageable batches
    id_response = arcgis_request(
        f"{BEACH_URL}/query",
        {
            "where": "1=1",
            "returnIdsOnly": "true",
            "f": "json"
        }
    )

    object_ids = sorted(id_response.get("objectIds") or [])

    if not object_ids:
        raise ValueError("The beach API returned no record IDs.")

    features = []

    for start in range(0, len(object_ids), 500):
        batch_ids = object_ids[start:start + 500]

        batch = arcgis_request(
            f"{BEACH_URL}/query",
            {
                "objectIds": ",".join(map(str, batch_ids)),
                "outFields": "*",
                "returnGeometry": "true",
                "outSR": 4326,
                "f": "geojson"
            }
        )

        if batch.get("exceededTransferLimit"):
            raise RuntimeError("API response was truncated. Reduce batch size.")

        features.extend(batch["features"])

    beach_access = gpd.GeoDataFrame.from_features(
        features,
        crs="EPSG:4326"
    )

    # Verify IDs as well as the number of records
    id_field = id_response["objectIdFieldName"]
    received_ids = set(beach_access[id_field])

    if (
        received_ids != set(object_ids)
        or len(beach_access) != len(object_ids)
    ):
        raise ValueError("Retrieved records do not match the requested IDs.")

    print(f"Retrieved {len(beach_access):,} coastal-access records.")
    print("Columns:", beach_access.columns.tolist())

    display(beach_access.head(3))

In [109]:
if REBUILD_AMENITIES:
    display(
        beach_access["ACESS_TYPE"]
        .value_counts(dropna=False)
        .rename_axis("access_type")
        .reset_index(name="records")
    )

    print(
        "Missing or empty geometries:",
        (
            beach_access.geometry.isna()
            | beach_access.geometry.is_empty
        ).sum()
    )

In [110]:
if REBUILD_AMENITIES:
    beach_access_clean = beach_access.copy()

    beach_access_clean["access_type_clean"] = (
        beach_access_clean["ACESS_TYPE"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
        .fillna("Unknown")
    )

    # Remove unusable locations
    usable_geometry = (
        beach_access_clean.geometry.notna()
        & ~beach_access_clean.geometry.is_empty
    )

    print(f"Records without usable geometry: {(~usable_geometry).sum():,}")

    beach_access_clean = (
        beach_access_clean.loc[usable_geometry]
        .copy()
        .to_crs("EPSG:32119")
    )

    # Separate subsets for later distance calculations
    beach_points = beach_access_clean[
        beach_access_clean["access_type_clean"].eq("Beach")
    ].copy()

    shoreline_points = beach_access_clean[
        beach_access_clean["access_type_clean"].eq("Shoreline")
    ].copy()

    beach_access_clean.to_file(
        OUTPUT_DIR / "nc_coastal_access_clean.gpkg",
        layer="coastal_access",
        driver="GPKG",
        engine="pyogrio"
    )

    print("Saved coastal-access data.")

In [111]:
if REBUILD_AMENITIES:
    RAIL_URL = (
        "https://gis11.services.ncdot.gov/arcgis/rest/services/"
        "NCDOT_RailSystem/MapServer/0"
    )

    # Get all facility IDs
    rail_id_response = arcgis_request(
        f"{RAIL_URL}/query",
        {
            "where": "1=1",
            "returnIdsOnly": "true",
            "f": "json"
        }
    )

    rail_ids = sorted(rail_id_response.get("objectIds") or [])

    if not rail_ids:
        raise ValueError("The rail API returned no record IDs.")

    rail_features = []

    for start in range(0, len(rail_ids), 100):
        batch_ids = rail_ids[start:start + 100]

        batch = arcgis_request(
            f"{RAIL_URL}/query",
            {
                "objectIds": ",".join(map(str, batch_ids)),
                "outFields": "*",
                "returnGeometry": "true",
                "outSR": 4326,
                "f": "geojson"
            }
        )

        if batch.get("exceededTransferLimit"):
            raise RuntimeError("Rail response was truncated.")

        rail_features.extend(batch["features"])

    rail_facilities = gpd.GeoDataFrame.from_features(
        rail_features,
        crs="EPSG:4326"
    )

    # Confirm that every requested record was retrieved exactly once
    rail_id_field = rail_id_response["objectIdFieldName"]

    if (
        set(rail_facilities[rail_id_field]) != set(rail_ids)
        or len(rail_facilities) != len(rail_ids)
    ):
        raise ValueError("Retrieved rail records do not match requested IDs.")

    print(f"Retrieved {len(rail_facilities):,} rail facilities.")

    display(
        rail_facilities["FacType"]
        .value_counts(dropna=False)
        .rename_axis("facility_type")
        .reset_index(name="records")
    )

In [112]:
if REBUILD_AMENITIES:
    # Select station candidates from the downloaded rail facilities
    station_mask = (
        rail_facilities["FacType"]
        .astype("string")
        .str.contains("Station", case=False, na=False)
    )

    rail_station_candidates = rail_facilities.loc[station_mask].copy()

    display(
        rail_station_candidates[
            ["LocationNa", "FacType", "Railroad", "County"]
        ].sort_values(["FacType", "LocationNa"])
    )

In [113]:
if REBUILD_AMENITIES:
    facility_type = (
        rail_station_candidates["FacType"]
        .astype("string")
        .str.strip()
    )

    rail_stations = rail_station_candidates[
        facility_type.eq("Station")
    ].copy()

    amtrak_bus_stops = rail_station_candidates[
        facility_type.eq("Station| Amtrak Bus")
    ].copy()

    rail_stations["access_mode"] = "passenger_rail"
    amtrak_bus_stops["access_mode"] = "amtrak_bus"

    transport_access = gpd.GeoDataFrame(
        pd.concat(
            [rail_stations, amtrak_bus_stops],
            ignore_index=True
        ),
        geometry="geometry",
        crs=rail_facilities.crs
    )

    # Check locations before saving
    usable = (
        transport_access.geometry.notna()
        & ~transport_access.geometry.is_empty
    )

    print(f"Records without usable geometry: {(~usable).sum():,}")

    transport_access = (
        transport_access.loc[usable]
        .copy()
        .to_crs("EPSG:32119")
    )

    transport_access.to_file(
        OUTPUT_DIR / "nc_rail_and_bus_access.gpkg",
        layer="transport_access",
        driver="GPKG",
        engine="pyogrio"
    )

    display(
        transport_access.groupby("access_mode")
        .size()
        .reset_index(name="records")
    )

In [114]:
if REBUILD_AMENITIES:
    osm_transport_points = gpd.read_file(
        OSM_PATH,
        layer="gis_osm_transport_free",
        engine="pyogrio"
    )

    osm_transport_areas = gpd.read_file(
        OSM_PATH,
        layer="gis_osm_transport_a_free",
        engine="pyogrio"
    )

    transport_categories = pd.concat(
        [
            osm_transport_points["fclass"]
            .value_counts().rename("points"),

            osm_transport_areas["fclass"]
            .value_counts().rename("areas")
        ],
        axis=1
    ).fillna(0).astype(int)

    print(transport_categories.to_string())

In [115]:
if REBUILD_AMENITIES:
    airport_parts = []

    for source_layer, df in [
        ("gis_osm_transport_free", osm_transport_points),
        ("gis_osm_transport_a_free", osm_transport_areas)
    ]:
        subset = df[
            df["fclass"].isin(["airport", "airfield"])
        ].copy()

        subset["source_layer"] = source_layer
        airport_parts.append(subset.to_crs("EPSG:32119"))

    airport_inventory = gpd.GeoDataFrame(
        pd.concat(airport_parts, ignore_index=True),
        geometry="geometry",
        crs="EPSG:32119"
    )

    # Compact list of distinct names for review
    airport_names = (
        airport_inventory["name"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
        .dropna()
        .drop_duplicates()
        .sort_values()
    )

    print(f"Airport/airfield records: {len(airport_inventory):,}")
    print("\nDistinct airport names:")
    print(airport_names.head(30).to_string(index=False))
    print("Showing at most 30 names; the full airport_inventory remains available.")

In [116]:
if REBUILD_AMENITIES:
    # Name patterns for NC's commercial passenger airport shortlist
    airport_patterns = {
        "Charlotte Douglas": r"charlotte.*douglas",
        "Raleigh-Durham": r"raleigh.*durham",
        "Piedmont Triad": r"piedmont.*triad",
        "Asheville": r"asheville.*regional",
        "Wilmington": r"wilmington.*international",
        "Fayetteville": r"fayetteville.*regional",
        "Albert J. Ellis": r"albert.*ellis",
        "Coastal Carolina": r"coastal.*carolina",
        "Pitt-Greenville": r"pitt.*greenville",
        "Concord-Padgett": r"concord.*(?:padgett|regional)"
    }

    airport_matches = []

    for airport, pattern in airport_patterns.items():
        matches = airport_inventory[
            airport_inventory["name"]
            .astype("string")
            .str.contains(pattern, case=False, regex=True, na=False)
        ].copy()

        matches["airport_group"] = airport
        airport_matches.append(matches)

    commercial_airport_candidates = gpd.GeoDataFrame(
        pd.concat(airport_matches, ignore_index=True),
        geometry="geometry",
        crs=airport_inventory.crs
    )

    # Small summary that won't flood the output
    summary = pd.DataFrame({
        "airport": list(airport_patterns),
        "matching_osm_records": [
            commercial_airport_candidates["airport_group"].eq(name).sum()
            for name in airport_patterns
        ]
    })

    display(summary)

In [117]:
if REBUILD_AMENITIES:
    padus_open.to_file(
        OUTPUT_DIR / "padus_open_clean.gpkg",
        layer="open_access_land",
        driver="GPKG",
        engine="pyogrio"
    )

    print(f"Saved {len(padus_open):,} open-access land records.")

In [118]:
if REBUILD_AMENITIES:
    commercial_airport_candidates.to_file(
        OUTPUT_DIR / "nc_airport_candidates.gpkg",
        layer="airport_candidates",
        driver="GPKG",
        engine="pyogrio"
    )

    print(f"Saved {len(commercial_airport_candidates):,} airport candidate records.")

## 5. Housing geography and integration

Load the prepared housing tables, then either load saved area polygons or rebuild them. Source: `data_intergration.ipynb`.

A tract/ZCTA intersection defines each final housing area. EPA's 2010 boundaries can fragment those polygons, so temporary overlap pieces are used only to select the largest-overlap EPA tract. The final dataset retains the original tract/ZCTA shapes, the EPA overlap share, and a flag below 90% coverage. No housing values are estimated at a finer resolution than their original source measurement.

In [119]:
# Read geographic IDs as text to preserve leading zeros
acs_nc = pd.read_csv(
    PROCESSED_DIR / "nc_acs_2024_tract_clean.csv",
    dtype={
        "GEOID": "string",
        "state": "string",
        "county": "string",
        "tract": "string"
    }
)

epa_nc = pd.read_csv(
    PROCESSED_DIR / "nc_epa_2010_tract_clean.csv",
    dtype={"tract": "string"}
)

zillow_nc = pd.read_csv(
    PROCESSED_DIR / "nc_zillow_zip_clean.csv",
    dtype={"zip": "string"}
)

housing_zip_nc = pd.read_csv(
    PROCESSED_DIR / "nc_zillow_epa_zip_clean.csv",
    dtype={"zip": "string"}
)

# Check row counts and unique geographic IDs
tables = {
    "ACS 2024 tracts": (acs_nc, "GEOID"),
    "EPA 2010 tracts": (epa_nc, "tract"),
    "Zillow ZIPs": (zillow_nc, "zip"),
    "Zillow + EPA ZIPs": (housing_zip_nc, "zip")
}

checks = []

for name, (df, id_col) in tables.items():
    checks.append({
        "dataset": name,
        "rows": len(df),
        "missing_ids": df[id_col].isna().sum(),
        "duplicate_ids": df[id_col].duplicated().sum()
    })

display(pd.DataFrame(checks))

for name, (df, _) in tables.items():
    print(f"\n{name} columns:")
    print(df.columns.tolist())

,dataset,rows,missing_ids,duplicate_ids
0,ACS 2024 tracts,2672,0,0
1,EPA 2010 tracts,2171,0,0
2,Zillow ZIPs,719,0,0
3,Zillow + EPA ZIPs,717,0,0



ACS 2024 tracts columns:
['GEOID', 'NAME', 'state', 'county', 'tract', 'median_hh_income', 'median_gross_rent', 'rent_pct_income', 'median_home_value', 'median_monthly_housing_cost', 'median_hh_income_moe', 'total_population', 'occupied_units', 'owner_units', 'renter_units']

EPA 2010 tracts columns:
['tract', 'D1A', 'D1C', 'D2B_E8MIXA', 'D3B', 'D5AR', 'NatWalkInd', 'Pct_AO0', 'R_PCTLOWWAGE', 'VMT_per_worker', 'has_transit']

Zillow ZIPs columns:
['zip', 'State', 'City', 'Metro', 'CountyName', 'home_value', 'home_value_12mo']

Zillow + EPA ZIPs columns:
['zip', 'State', 'City', 'Metro', 'CountyName', 'home_value', 'home_value_12mo', 'D1A', 'D1C', 'D2B_E8MIXA', 'D3B', 'D5AR', 'NatWalkInd', 'Pct_AO0', 'R_PCTLOWWAGE', 'VMT_per_worker', 'has_transit']


In [120]:
for name, (frame, identifier) in tables.items():
    expected_length = 5 if identifier == 'zip' else 11
    valid = frame[identifier].astype('string').str.fullmatch(rf'\d{{{expected_length}}}', na=False)
    if not valid.all() or frame[identifier].duplicated().any():
        raise ValueError(f'{name}: IDs must be unique {expected_length}-digit strings.')
if not REBUILD_AREAS:
    housing_epa_areas = gpd.read_file(
        PROCESSED_DIR / 'nc_housing_epa_integrated.gpkg', layer='housing_epa', engine='pyogrio'
    ).to_crs(PROJECT_CRS)
    print(f'Loaded {len(housing_epa_areas):,} saved housing areas.')

Loaded 5,466 saved housing areas.


In [121]:
if REBUILD_AREAS:
    import requests

    BOUNDARY_DIR = DATA_DIR / "raw" / "boundaries"
    BOUNDARY_DIR.mkdir(parents=True, exist_ok=True)

    boundary_urls = {
        "tracts": (
            "https://www2.census.gov/geo/tiger/"
            "TIGER2024/TRACT/tl_2024_37_tract.zip"
        ),
        "zctas": (
            "https://www2.census.gov/geo/tiger/"
            "TIGER2024/ZCTA520/tl_2024_us_zcta520.zip"
        )
    }

    boundary_paths = {}

    for name, url in boundary_urls.items():
        path = BOUNDARY_DIR / url.rsplit("/", 1)[-1]

        if not path.exists():
            print(f"Downloading {name}...")
            temporary_path = path.with_suffix(".part")

            with requests.get(url, stream=True, timeout=(30, 180)) as response:
                response.raise_for_status()

                with temporary_path.open("wb") as file:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            file.write(chunk)

            temporary_path.replace(path)

        boundary_paths[name] = path

    tract_boundaries = gpd.read_file(
        f"zip://{boundary_paths['tracts'].resolve().as_posix()}",
        engine="pyogrio"
    )

    # Both source files use the same coordinate system.
    # Read only ZCTAs within NC's bounding box, including border overlaps.
    zcta_boundaries = gpd.read_file(
        f"zip://{boundary_paths['zctas'].resolve().as_posix()}",
        bbox=tuple(tract_boundaries.total_bounds),
        engine="pyogrio"
    )

    print(f"NC tract boundaries: {len(tract_boundaries):,}")
    print(f"Nearby ZCTA boundaries: {len(zcta_boundaries):,}")

In [122]:
if REBUILD_AREAS:
    # Keep the IDs and geometry
    tract_shapes = tract_boundaries[
        ["GEOID", "geometry"]
    ].copy()

    zcta_shapes = zcta_boundaries[
        ["ZCTA5CE20", "geometry"]
    ].rename(columns={"ZCTA5CE20": "zcta"}).copy()

    tract_shapes["GEOID"] = tract_shapes["GEOID"].astype("string")
    zcta_shapes["zcta"] = zcta_shapes["zcta"].astype("string")

    # Use NC State Plane so area calculations are in square meters
    tract_shapes = tract_shapes.to_crs("EPSG:32119")
    zcta_shapes = zcta_shapes.to_crs("EPSG:32119")

    # Create a polygon for each tract–ZCTA overlap
    overlap_areas = gpd.overlay(
        tract_shapes,
        zcta_shapes,
        how="intersection",
        keep_geom_type=True
    )

    # Group disconnected pieces with the same tract and ZCTA IDs
    overlap_areas = overlap_areas.dissolve(
        by=["GEOID", "zcta"],
        as_index=False
    )

    overlap_areas["area_id"] = (
        overlap_areas["GEOID"] + "_" + overlap_areas["zcta"]
    )

    overlap_areas["area_sq_km"] = (
        overlap_areas.geometry.area / 1_000_000
    )

    print(f"Created {len(overlap_areas):,} tract–ZCTA areas.")

    display(
        overlap_areas[
            ["area_id", "GEOID", "zcta", "area_sq_km"]
        ].head()
    )

In [123]:
if REBUILD_AREAS:
    acs_attributes = acs_nc.rename(
        columns={c: f"acs_{c}" for c in acs_nc.columns if c != "GEOID"}
    )

    zillow_attributes = zillow_nc.rename(
        columns={c: f"zillow_{c}" for c in zillow_nc.columns if c != "zip"}
    )

    integrated_areas = overlap_areas.merge(
        acs_attributes,
        on="GEOID",
        how="left",
        validate="many_to_one",
        indicator="acs_match"
    )

    integrated_areas = integrated_areas.merge(
        zillow_attributes,
        left_on="zcta",
        right_on="zip",
        how="left",
        validate="many_to_one",
        indicator="zillow_match"
    )

    display(
        integrated_areas.groupby(
            ["acs_match", "zillow_match"],
            observed=True
        ).size().reset_index(name="areas")
    )

    # Convert merge indicators to text before saving
    for column in ["acs_match", "zillow_match"]:
        integrated_areas[column] = integrated_areas[column].astype("string")

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    integrated_areas.to_file(
        PROCESSED_DIR / "nc_tract_zcta_housing.gpkg",
        layer="tract_zcta_housing",
        driver="GPKG",
        engine="pyogrio"
    )

    print(f"Saved {len(integrated_areas):,} integrated areas.")

In [124]:
if REBUILD_AREAS:
    display(
        integrated_areas["area_sq_km"]
        .describe(percentiles=[0.01, 0.05, 0.5, 0.95])
        .to_frame("area_sq_km")
    )

    print(
        "Areas smaller than 1,000 square meters:",
        integrated_areas["area_sq_km"].lt(0.001).sum()
    )

    print(
        "Unique ACS tracts represented:",
        integrated_areas["GEOID"].nunique()
    )

    print(
        "Unique ZCTAs represented:",
        integrated_areas["zcta"].nunique()
    )

In [125]:
if REBUILD_AREAS:
    # Find ACS tracts that have no polygon in the overlap table
    missing_tracts = acs_nc.loc[
        ~acs_nc["GEOID"].isin(integrated_areas["GEOID"]),
        ["GEOID", "NAME", "total_population", "occupied_units"]
    ].copy()

    display(missing_tracts)

In [126]:
if REBUILD_AREAS:
    # Download NC's 2010 census tract boundaries
    epa_boundary_url = (
        "https://www2.census.gov/geo/tiger/"
        "TIGER2010/TRACT/2010/tl_2010_37_tract10.zip"
    )

    epa_boundary_path = BOUNDARY_DIR / "tl_2010_37_tract10.zip"

    if not epa_boundary_path.exists():
        temporary_path = epa_boundary_path.with_suffix(".part")

        with requests.get(
            epa_boundary_url,
            stream=True,
            timeout=(30, 180)
        ) as response:
            response.raise_for_status()

            with temporary_path.open("wb") as file:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        file.write(chunk)

        temporary_path.replace(epa_boundary_path)

    tracts_2010 = gpd.read_file(
        f"zip://{epa_boundary_path.resolve().as_posix()}",
        engine="pyogrio"
    )

    # Keep the older tract ID distinct from the ACS tract ID
    epa_shapes = (
        tracts_2010[["GEOID10", "geometry"]]
        .rename(columns={"GEOID10": "epa_tract"})
        .to_crs(integrated_areas.crs)
    )

    epa_shapes["epa_tract"] = epa_shapes["epa_tract"].astype("string")

    # Check whether every EPA record has a matching boundary
    unmatched_epa = epa_nc.loc[
        ~epa_nc["tract"].isin(epa_shapes["epa_tract"])
    ]

    print(f"2010 tract boundaries: {len(epa_shapes):,}")
    print(f"EPA records: {len(epa_nc):,}")
    print(f"EPA records without a boundary: {len(unmatched_epa):,}")

    if not unmatched_epa.empty:
        display(unmatched_epa[["tract"]])

In [127]:
if REBUILD_AREAS:
    # Prefix EPA variables and preserve its original tract ID
    epa_attributes = epa_nc.rename(
        columns={
            c: "epa_tract" if c == "tract" else f"epa_{c}"
            for c in epa_nc.columns
        }
    )

    epa_geography = epa_shapes.merge(
        epa_attributes,
        on="epa_tract",
        how="left",
        validate="one_to_one",
        indicator="epa_match"
    )

    epa_geography["epa_match"] = (
        epa_geography["epa_match"].astype("string")
    )

    # Keep the original area ID as a reference.
    # Recalculate area after splitting.
    base_areas = integrated_areas.rename(
        columns={"area_id": "parent_area_id"}
    ).drop(columns=["area_sq_km"])

    # Identity preserves portions without a matching 2010 boundary
    integrated_epa = gpd.overlay(
        base_areas,
        epa_geography,
        how="identity",
        keep_geom_type=True
    )

    integrated_epa["epa_match"] = (
        integrated_epa["epa_match"].fillna("no_boundary")
    )

    # Each row is a polygon piece with its source-area attributes
    integrated_epa = integrated_epa.reset_index(drop=True)

    integrated_epa["area_id"] = (
        "NC_" + integrated_epa.index.astype(str).str.zfill(7)
    )

    integrated_epa["area_sq_km"] = (
        integrated_epa.geometry.area / 1_000_000
    )

    print(f"Original tract–ZCTA areas: {len(integrated_areas):,}")
    print(f"Areas after EPA overlay: {len(integrated_epa):,}")

    display(
        integrated_epa.groupby("epa_match")
        .agg(
            areas=("area_id", "size"),
            total_sq_km=("area_sq_km", "sum")
        )
        .reset_index()
    )

    print(
        "Original areas preserved:",
        integrated_epa["parent_area_id"].nunique(),
        "out of",
        integrated_areas["area_id"].nunique()
    )

    print(
        "Pieces smaller than 1,000 square meters:",
        integrated_epa["area_sq_km"].lt(0.001).sum()
    )

In [128]:
if REBUILD_AREAS:
    # Total overlapping area for each original area and EPA tract
    epa_overlap = (
        integrated_epa.dropna(subset=["epa_tract"])
        .groupby(["parent_area_id", "epa_tract"], as_index=False)
        .agg(epa_overlap_sq_km=("area_sq_km", "sum"))
    )

    # Select the EPA tract covering the largest portion of each area
    dominant_epa = (
        epa_overlap
        .sort_values(
            ["parent_area_id", "epa_overlap_sq_km", "epa_tract"],
            ascending=[True, False, True]
        )
        .drop_duplicates("parent_area_id")
        .rename(columns={"parent_area_id": "area_id"})
    )

    # Keep the original polygons and attach the selected EPA tract
    housing_epa_areas = integrated_areas.merge(
        dominant_epa,
        on="area_id",
        how="left",
        validate="one_to_one"
    )

    housing_epa_areas["epa_overlap_share"] = (
        housing_epa_areas["epa_overlap_sq_km"]
        / housing_epa_areas["area_sq_km"]
    )

    housing_epa_areas = housing_epa_areas.merge(
        epa_attributes,
        on="epa_tract",
        how="left",
        validate="many_to_one",
        indicator="epa_data_match"
    )

    print(f"Areas retained: {len(housing_epa_areas):,}")

    display(
        housing_epa_areas["epa_overlap_share"]
        .describe(percentiles=[0.05, 0.5, 0.95])
        .to_frame()
    )

    print(
        "Areas with less than 90% dominant EPA coverage:",
        housing_epa_areas["epa_overlap_share"].lt(0.90).sum()
    )

    display(
        housing_epa_areas["epa_data_match"]
        .value_counts()
        .rename_axis("match")
        .reset_index(name="areas")
    )

In [129]:
if REBUILD_AREAS:
    # Flag areas where the selected EPA tract covers less than 90%
    housing_epa_areas["epa_low_overlap"] = (
        housing_epa_areas["epa_overlap_share"].lt(0.90)
    )

    housing_epa_areas["epa_assignment_method"] = "largest_area_overlap"

    # Convert the merge indicator before saving
    housing_epa_areas["epa_data_match"] = (
        housing_epa_areas["epa_data_match"].astype("string")
    )

    housing_epa_areas.to_file(
        PROCESSED_DIR / "nc_housing_epa_integrated.gpkg",
        layer="housing_epa",
        driver="GPKG",
        engine="pyogrio"
    )

    print(f"Saved {len(housing_epa_areas):,} integrated areas.")

In [130]:
if housing_epa_areas['area_id'].isna().any() or housing_epa_areas['area_id'].duplicated().any():
    raise ValueError('Housing area IDs must be nonmissing and unique.')
if housing_epa_areas.empty:
    raise ValueError('The housing area dataset is empty.')
missing_tracts = acs_nc.loc[~acs_nc['GEOID'].isin(housing_epa_areas['GEOID']),
                          ['GEOID', 'NAME', 'total_population', 'occupied_units']]
print('ACS tracts absent from the final areas:', len(missing_tracts))
display(missing_tracts)
coverage = pd.DataFrame([
    {'source': label, 'matched_areas': int(housing_epa_areas[col].eq('both').sum()),
     'total_areas': len(housing_epa_areas)}
    for label, col in [('ACS', 'acs_match'), ('Zillow', 'zillow_match'), ('EPA', 'epa_data_match')]
])
coverage['matched_pct'] = 100 * coverage['matched_areas'] / coverage['total_areas']
display(coverage)
display(housing_epa_areas['epa_overlap_share'].describe(percentiles=[.05, .5, .95]).to_frame())

ACS tracts absent from the final areas: 11


,GEOID,NAME,total_population,occupied_units
327,37031980100,Census Tract 9801; Carteret County; North Caro...,0,0
328,37031990100,Census Tract 9901; Carteret County; North Caro...,0,0
329,37031990200,Census Tract 9902; Carteret County; North Caro...,0,0
564,37053990100,Census Tract 9901; Currituck County; North Car...,0,0
575,37055990100,Census Tract 9901; Dare County; North Carolina,0,0
576,37055990200,Census Tract 9902; Dare County; North Carolina,0,0
1197,37095990100,Census Tract 9901; Hyde County; North Carolina,0,0
1198,37095990200,Census Tract 9902; Hyde County; North Carolina,0,0
1823,37129990100,Census Tract 9901; New Hanover County; North C...,0,0
1928,37137990100,Census Tract 9901; Pamlico County; North Carolina,0,0


,source,matched_areas,total_areas,matched_pct
0,ACS,5466,5466,100.000000
1,Zillow,5279,5466,96.578851
2,EPA,5442,5466,99.560922


,epa_overlap_share
count,5466.000000
mean,0.994525
std,0.029053
min,0.511618
5%,0.981325
50%,0.999958
95%,1.000000
max,1.000000


## 6. Accessibility features

Every housing area uses one representative point inside its polygon. Amenity polygons also use one representative point per mapped record. These reference points need not be entrances or places where people live.

Counts use a fixed 5 km search radius by default and can include the same amenity in several housing areas. Nearest-access distances use the full destination geometry: for polygon airports, the result is distance to the mapped property boundary, not a passenger terminal or driving route.

In [131]:
amenity_files = {
    "osm_amenities": ("osm_amenities_clean.gpkg", "amenities"),
    "padus_open": ("padus_open_clean.gpkg", "open_access_land"),
    "coastal_access": ("nc_coastal_access_clean.gpkg", "coastal_access"),
    "transport_access": ("nc_rail_and_bus_access.gpkg", "transport_access"),
    "airport_candidates": ("nc_airport_candidates.gpkg", "airport_candidates")
}

amenity_tables = {}

for name, (filename, layer) in amenity_files.items():
    df = gpd.read_file(
        PROCESSED_DIR / filename,
        layer=layer,
        engine="pyogrio"
    )

    amenity_tables[name] = df.to_crs(housing_epa_areas.crs)

display(
    pd.DataFrame([
        {
            "dataset": name,
            "records": len(df),
            "missing_or_empty_geometry": (
                df.geometry.isna() | df.geometry.is_empty
            ).sum(),
            "invalid_geometry": (
                df.geometry.notna()
                & ~df.geometry.is_empty
                & ~df.geometry.is_valid
            ).sum()
        }
        for name, df in amenity_tables.items()
    ])
)

# Inspect the groups available for amenity features
osm_amenities = amenity_tables["osm_amenities"]

display(
    osm_amenities["amenity_group"]
    .value_counts(dropna=False)
    .rename_axis("amenity_group")
    .reset_index(name="records")
)

,dataset,records,missing_or_empty_geometry,invalid_geometry
0,osm_amenities,60273,0,0
1,padus_open,2830,0,0
2,coastal_access,911,0,0
3,transport_access,27,0,0
4,airport_candidates,9,0,0


,amenity_group,records
0,sports_and_fitness,27656
1,food_and_nightlife,15370
2,outdoor_recreation,10071
3,shopping_and_daily_needs,5795
4,entertainment_and_culture,1381


In [132]:
padus_open = amenity_tables["padus_open"].copy()

invalid = ~padus_open.geometry.is_valid
padus_open.loc[invalid, "geometry"] = (
    padus_open.loc[invalid, "geometry"].make_valid()
)

amenity_tables["padus_open"] = padus_open

print("Invalid PAD-US geometries remaining:",
      (~padus_open.geometry.is_valid).sum())

Invalid PAD-US geometries remaining: 0


In [133]:
# Confirm geometry before the spatial joins. Repair invalid shapes explicitly.
geometry_checks = []
for name, frame in {'housing_areas': housing_epa_areas, **amenity_tables}.items():
    if frame.crs is None:
        raise ValueError(f'{name}: missing coordinate system.')
    unusable = frame.geometry.isna() | frame.geometry.is_empty
    if unusable.any():
        raise ValueError(f'{name}: {unusable.sum()} missing/empty geometries.')
    invalid = ~frame.geometry.is_valid
    if invalid.any():
        frame.loc[invalid, 'geometry'] = frame.loc[invalid, 'geometry'].make_valid()
    if frame.geometry.isna().any() or frame.geometry.is_empty.any() or (~frame.geometry.is_valid).any():
        raise ValueError(f'{name}: geometry still unusable after repair.')
    geometry_checks.append({'dataset': name, 'records': len(frame), 'repaired_geometries': int(invalid.sum())})
display(pd.DataFrame(geometry_checks))
# Persist the repaired PAD-US copy as part of the cleaned outputs.
save_gpkg(amenity_tables['padus_open'], 'padus_open_clean.gpkg', 'open_access_land')

,dataset,records,repaired_geometries
0,housing_areas,5466,0
1,osm_amenities,60273,0
2,padus_open,2830,0
3,coastal_access,911,0
4,transport_access,27,0
5,airport_candidates,9,0


In [134]:
# Use one point inside each housing area.
# This is a reference location, not necessarily where residents live.
area_points = housing_epa_areas[["area_id", "geometry"]].copy()
area_points["geometry"] = area_points.geometry.representative_point()

# Represent each mapped amenity record with one point
amenity_points = osm_amenities[
    ["amenity_group", "geometry"]
].copy()

amenity_points["geometry"] = (
    amenity_points.geometry.representative_point()
)

# Create a 5 km search area around each housing reference point
search_areas = area_points.copy()
search_areas["geometry"] = search_areas.geometry.buffer(SEARCH_RADIUS_M)

nearby = gpd.sjoin(
    amenity_points,
    search_areas,
    how="inner",
    predicate="intersects"
)

# Count mapped records by area and amenity group
groups = sorted(amenity_points["amenity_group"].dropna().unique())

amenity_counts = (
    nearby.groupby(["area_id", "amenity_group"])
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=housing_epa_areas["area_id"],
        columns=groups,
        fill_value=0
    )
    .fillna(0)
    .astype(int)
)

amenity_counts.columns = [
    f"osm_{group}_within_{SEARCH_RADIUS_M / 1000:g}km"
    for group in amenity_counts.columns
]

amenity_counts = amenity_counts.reset_index()

display(
    amenity_counts.drop(columns="area_id")
    .describe()
    .round(1)
)

,osm_entertainment_and_culture_within_5km,osm_food_and_nightlife_within_5km,osm_outdoor_recreation_within_5km,osm_shopping_and_daily_needs_within_5km,osm_sports_and_fitness_within_5km
count,5466.0,5466.0,5466.0,5466.0,5466.0
mean,3.6,52.3,25.4,14.5,70.1
std,7.5,100.7,39.9,20.2,98.7
min,0.0,0.0,0.0,0.0,0.0
25%,0.0,1.0,1.0,1.0,4.0
50%,1.0,11.0,8.0,5.0,26.0
75%,3.0,58.0,32.8,21.0,97.0
max,51.0,742.0,246.0,117.0,564.0


In [135]:
housing_features = housing_epa_areas.merge(
    amenity_counts,
    on="area_id",
    how="left",
    validate="one_to_one"
)

print(f"Areas with amenity features: {len(housing_features):,}")

Areas with amenity features: 5,466


In [136]:
coastal = amenity_tables["coastal_access"]
transport = amenity_tables["transport_access"]
airports = amenity_tables["airport_candidates"]

destinations = {
    "beach_access": coastal.loc[
        coastal["access_type_clean"].eq("Beach")
    ],
    "shoreline_access": coastal.loc[
        coastal["access_type_clean"].eq("Shoreline")
    ],
    "passenger_rail": transport.loc[
        transport["access_mode"].eq("passenger_rail")
    ],
    "amtrak_bus": transport.loc[
        transport["access_mode"].eq("amtrak_bus")
    ],
    "airport_candidate": airports
}

# Reuse the housing reference points from the amenity-count cell
for name, locations in destinations.items():
    if locations.empty:
        raise ValueError(f"No locations available for {name}.")

    nearest = gpd.sjoin_nearest(
        area_points[["area_id", "geometry"]],
        locations[["geometry"]],
        how="left",
        distance_col="distance_m"
    )

    # Equally near destinations can produce multiple matches
    distances_km = (
        nearest.groupby("area_id")["distance_m"].min() / 1000
    )

    housing_features[f"nearest_{name}_km"] = (
        housing_features["area_id"].map(distances_km)
    )

distance_columns = [
    f"nearest_{name}_km" for name in destinations
]

display(
    housing_features[distance_columns]
    .describe()
    .round(2)
)

,nearest_beach_access_km,nearest_shoreline_access_km,nearest_passenger_rail_km,nearest_amtrak_bus_km,nearest_airport_candidate_km
count,5466.00,5466.00,5466.00,5466.00,5466.00
mean,222.56,196.82,49.92,88.84,47.54
std,111.10,123.36,49.42,60.17,33.78
min,0.07,0.04,0.08,0.13,0.00
25%,144.31,106.12,13.10,45.39,18.29
50%,236.22,205.06,30.29,85.30,41.24
75%,290.46,276.59,72.00,114.76,70.75
max,543.48,543.40,282.17,380.47,160.28


## 7. Validate, save, and visualize the combined dataset

`project_data` is the final GeoDataFrame for subsequent analysis. It contains source identifiers and match flags, housing and EPA attributes, amenity counts, and five nearest-access distances. The CSV omits geometry; use the GeoPackage for maps and spatial analysis. Re-running these export cells replaces the named outputs.

In [137]:
project_data = housing_features.copy()
count_columns = [c for c in project_data if c.startswith('osm_') and '_within_' in c]
if len(project_data) != len(housing_epa_areas) or project_data['area_id'].duplicated().any():
    raise ValueError('Accessibility joins changed the area count or uniqueness.')
if project_data[count_columns].isna().any().any():
    raise ValueError('Unexpected missing amenity counts.')
if project_data[distance_columns].isna().any().any():
    raise ValueError('At least one nearest-access distance is missing.')
if (project_data[count_columns + distance_columns] < 0).any().any():
    raise ValueError('Counts and distances must not be negative.')

save_gpkg(project_data, 'nc_housing_features.gpkg', 'housing_features')
project_data.drop(columns='geometry').to_csv(PROCESSED_DIR / 'nc_housing_features.csv', index=False)
amenity_counts.to_csv(PROCESSED_DIR / 'nc_amenity_counts.csv', index=False)
coverage.to_csv(PROCESSED_DIR / 'integration_coverage.csv', index=False)

notes = {
    'area_id': 'ACS tract GEOID plus ZCTA identifier; a polygon may have disconnected pieces.',
    'GEOID': 'ACS tract geographic identifier, not interchangeable with 2010 EPA tract IDs.',
    'zcta': 'Census approximation to ZIP service area; matched to Zillow ZIP identifier.',
    'epa_tract': '2010 tract selected by largest polygon overlap.',
    'epa_overlap_share': 'Fraction of final area covered by the selected EPA tract.',
    'epa_low_overlap': 'True when selected EPA coverage is below 90%.',
    'acs_rent_pct_income': 'Median gross rent as percentage of income; not share of households burdened.',
    'zillow_home_value_12mo': 'Mean of up to the last 12 months in supplied Zillow file.',
    'epa_has_transit': 'Population-weighted nonmissing-D4A proxy from original cleaning; verify interpretation.'
}
def column_note(column):
    if column in notes:
        return notes[column]
    if column in count_columns:
        return f'Mapped OSM records within {SEARCH_RADIUS_M} meters of housing reference point.'
    if column in distance_columns:
        return 'Straight-line kilometers to nearest mapped destination geometry; not driving distance.'
    if column.startswith('acs_'):
        return 'Original ACS 2020-2024 source-tract statistic, repeated across overlapping areas.'
    if column.startswith('zillow_'):
        return 'Original Zillow ZIP attribute, repeated across overlapping areas.'
    if column.startswith('epa_'):
        return 'EPA source-tract attribute or assignment metadata; not a new estimate for the smaller area.'
    return 'Geographic identifier, geometry, source match indicator, or source attribute.'
dictionary = pd.DataFrame([
    {'column': col, 'dtype': str(project_data[col].dtype),
     'missing_rows': int(project_data[col].isna().sum()), 'description': column_note(col)}
    for col in project_data.columns
])
dictionary.to_csv(PROCESSED_DIR / 'nc_housing_data_dictionary.csv', index=False)
print(f'Saved {len(project_data):,} rows and {len(project_data.columns)} columns to nc_housing_features.gpkg and .csv')
display(dictionary)

Saved 5,466 rows and 54 columns to nc_housing_features.gpkg and .csv


,column,dtype,missing_rows,description
0,GEOID,object,0,"ACS tract geographic identifier, not interchan..."
1,zcta,object,0,Census approximation to ZIP service area; matc...
2,area_id,object,0,ACS tract GEOID plus ZCTA identifier; a polygo...
3,area_sq_km,float64,0,"Geographic identifier, geometry, source match ..."
4,acs_NAME,object,0,"Original ACS 2020-2024 source-tract statistic,..."
5,acs_state,object,0,"Original ACS 2020-2024 source-tract statistic,..."
6,acs_county,object,0,"Original ACS 2020-2024 source-tract statistic,..."
7,acs_tract,object,0,"Original ACS 2020-2024 source-tract statistic,..."
8,acs_median_hh_income,float64,51,"Original ACS 2020-2024 source-tract statistic,..."
9,acs_median_gross_rent,float64,271,"Original ACS 2020-2024 source-tract statistic,..."


In [138]:
if DRAW_INTERACTIVE_MAP:
    import folium

    # Use a simplified copy for display.
    # The saved area dataset keeps its original geometry.
    map_areas = housing_epa_areas[
        ["area_id", "GEOID", "zcta", "epa_tract",
         "epa_overlap_share", "geometry"]
    ].copy()

    map_areas["epa_overlap_pct"] = (
        map_areas["epa_overlap_share"] * 100
    ).round(1)

    map_areas["geometry"] = map_areas.geometry.simplify(
        tolerance=20,
        preserve_topology=True
    )

    map_areas = map_areas.to_crs("EPSG:4326")

    area_map = folium.Map(
        location=[35.5, -79.8],
        zoom_start=7,
        tiles="CartoDB positron"
    )

    folium.GeoJson(
        map_areas.to_json(),
        name="Tract–ZCTA housing areas",
        style_function=lambda feature: {
            "fillColor": "#3182bd",
            "color": "#18466a",
            "weight": 0.7,
            "fillOpacity": 0.18
        },
        highlight_function=lambda feature: {
            "fillColor": "#ffb347",
            "fillOpacity": 0.6,
            "weight": 2
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[
                "area_id", "GEOID", "zcta",
                "epa_tract", "epa_overlap_pct"
            ],
            aliases=[
                "Area ID:",
                "ACS tract:",
                "ZCTA:",
                "Assigned EPA tract:",
                "EPA overlap (%):"
            ],
            localize=True
        )
    ).add_to(area_map)

    west, south, east, north = map_areas.total_bounds
    area_map.fit_bounds([[south, west], [north, east]])

    map_path = PROCESSED_DIR / "nc_integrated_areas_map.html"
    area_map.save(str(map_path))

    print(f"Map saved to: {map_path.resolve()}")

    display(area_map)

In [139]:
if DRAW_PRESENTATION_MAPS:
    window = gpd.GeoSeries([box(-81.05, 35.00, -80.55, 35.50)], crs=4326).to_crs(PROJECT_CRS)
    downtown = gpd.GeoSeries([Point(-80.8431, 35.2271)], crs=4326).to_crs(PROJECT_CRS).iloc[0]
    charlotte_areas = gpd.clip(project_data[['geometry']], window)
    fig, axes = plt.subplots(1, 2, figsize=(15, 7), gridspec_kw={'width_ratios': [1.5, 1]})
    project_data.plot(ax=axes[0], facecolor='#dceaf5', edgecolor='#254e70', linewidth=.18)
    xmin, ymin, xmax, ymax = window.total_bounds
    axes[0].add_patch(Rectangle((xmin, ymin), xmax-xmin, ymax-ymin, fill=False, edgecolor='#d76b29', linewidth=1.5))
    if not charlotte_areas.empty:
        charlotte_areas.plot(ax=axes[1], facecolor='#dceaf5', edgecolor='#254e70', linewidth=.65)
    axes[1].set_xlim(xmin, xmax)
    axes[1].set_ylim(ymin, ymax)
    axes[1].scatter(downtown.x, downtown.y, s=40, color='#d76b29', zorder=3)
    axes[1].annotate('Downtown Charlotte', (downtown.x, downtown.y), xytext=(8, 8), textcoords='offset points',
                     fontsize=9, bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': .9})
    for ax, title in zip(axes, ['North Carolina housing areas', 'Charlotte close-up']):
        ax.set_title(title, fontsize=16, fontweight='bold')
        ax.set_axis_off()
    fig.text(.04, .025, 'Census tract/ZCTA intersections. EPA attributes assigned by largest overlap. Source: Census boundaries.', fontsize=9)
    fig.tight_layout(rect=[0, .06, 1, 1])
    fig.savefig(FIG_DIR / 'housing_areas_nc_charlotte.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

In [140]:
if DRAW_PRESENTATION_MAPS:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(coverage['source'], [100]*len(coverage), color='#e8edf2')
    bars = ax.barh(coverage['source'], coverage['matched_pct'], color=['#37658a', '#409b90', '#8176ac'])
    for bar, row in zip(bars, coverage.itertuples()):
        ax.text(2, bar.get_y()+bar.get_height()/2, f'{row.matched_areas:,} areas ({row.matched_pct:.1f}%)', color='white', va='center')
    ax.set_xlim(0, 100)
    ax.invert_yaxis()
    ax.set_xlabel('Areas with matching source record (%)')
    ax.set_title('Geographic source coverage')
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'source_coverage.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 6))
    box_data = [project_data[col].dropna() for col in count_columns]
    ax.boxplot(box_data, patch_artist=True, flierprops={'markersize': 2, 'alpha': .15})
    ax.set_xticks(range(1, len(count_columns)+1))
    ax.set_xticklabels([c.removeprefix('osm_').split('_within_')[0].replace('_and_', ' &\n').replace('_', ' ') for c in count_columns], fontsize=9)
    ax.set_yscale('symlog', linthresh=1)
    ax.set_ylabel(f'Mapped amenities within {SEARCH_RADIUS_M / 1000:g} km (compressed scale)')
    ax.set_title('Nearby amenity counts vary across housing areas')
    fig.text(.02, .015, 'Reference-point straight-line searches; records may include multiple features per venue. Boxes show the middle 50%.', fontsize=9)
    fig.tight_layout(rect=[0, .07, 1, 1])
    fig.savefig(FIG_DIR / 'amenity_count_distributions.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## 8. Interpretation and next steps

This notebook completes preparation and initial descriptive features. It does **not** fit a predictive model or produce personalized recommendations.

- Source attributes describe their parent ZIPs or tracts. Intersecting boundaries does not add independent housing observations. Do not sum repeated parent-tract population or housing counts over the final polygons.
- Training/test splits must account for shared source geography and spatial dependence. For a ZIP-level Zillow target, keep all polygons sharing that ZIP together. Select the evaluation unit after choosing the outcome.
- ACS estimates have sampling uncertainty, top/bottom coding, and different household universes. Preserve missing values and review margins of error.
- OSM mapping completeness and public access vary. Counts are mapped records; fields and pools can be private. PAD-US polygons can overlap and do not identify entrances.
- EPA aggregation and largest-overlap assignment are approximations. Review low-overlap areas and the original `has_transit` proxy before modeling.
- Airport coverage remains incomplete if the shortlist is missing matches. Boundaries do not identify terminal entrances; station labels do not verify current service schedules.
- Datasets represent different years. Record source vintages before interpreting correlations or fitting models.

Next: inspect missingness, select an affordability or home-value outcome, explore associations at an appropriate source geography, build a baseline, and evaluate recommendations against defined user preferences.

### Consolidation notes

The notebook retains the four source workflows and their useful checks. Repeated setup, duplicate ID summaries, intermediate progress narratives, package-install cells, and the older Zillow map based on a separate 2010 ZIP download were replaced with a shared setup and current final-area maps. Original saved outputs were cleared. Final accessibility exports and a data dictionary were added so subsequent work can load one table.